### Install Libraries

In [1]:
!pip install ddgs trafilatura

### Setup

In [2]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()
MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [3]:
def search_web(query: str):
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 Got Results")
    return json.dumps(results, indent=2)

In [4]:
#'beautiful soup' is another package you can use if you need more control

def fetch_url(url: str):
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 Got text: {len(text)} chars")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

In [5]:
results = search_web("AI in healthcare in 2030")
print(results)

  ✅ Got Results
[
  {
    "title": "Pulse Report: Healthcare in 2030: How AI Agents Promote Integrated ...",
    "href": "https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/",
    "body": "With the rapid democratization of AI, the future of healthcare was already under redevelopment. Now, in an era of AI agents, that future is getting reshaped. What might this new healthcare future, powered by AI agents, look like? Imagine analyzing the most complex, distributed data across vast ecosystems to forecast possible health events or predict surges in demand, activating both mitigation ..."
  },
  {
    "title": "Future of AI in Healthcare: What Will Hospitals Look Like in 2030?",
    "href": "https://eyesforscience.com/future-of-ai-in-healthcare-2030/",
    "body": "AI is transforming every industry. Know about the future of AI in healthcare and how it will transform hospitals by 2030."
  },
  {
    "title": "AI a

In [6]:
fetch_results = fetch_url("https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/")
print(fetch_results)

  ❌ Failed to fetch or extract text from https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/
Could not extract text from https://www.himss.org/resources/pulse-report-healthcare-in-2030-how-ai-agents-promote-integrated-collaborative-and-proactive-care/. Try a different source.


### Step 2: Define as LLM Tools

In [7]:
tools = []

In [8]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using DuckDuckGo browser. Returns 3 results",
    "parameters" : {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant websites."
            }
        },
        "required": ["query"]
    }
}

fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch and extract the main text content from a web page",
    "parameters" : {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL of the web page to fetch and extract text from"
            }
        },
        "required": ["url"]
    }
}

tools.extend([{"type": "function", "function": fetch_url_function}, {"type": "function", "function": search_web_function}])

### Step 3: Tool Handler

In [17]:

def handle_tool_calls(tool_calls):
    tool_call_results = []

    for tool_call in tool_calls:
        function_name = tool_call.function.name
        if function_name == "search_web":
            args = json.loads(tool_call.function.arguments) 
            print(f"  \U0001f527 Calling function {function_name} with arguments: {args}") #for debugging
            result = search_web(args["query"])
            content = f"Search Result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f'"Fetched URL Content: {result}'
        else:
            content = f"Unknown tool call: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
            
        tool_call_results.append(tool_call_result)
        

    return tool_call_results

### Step 4: The System Prompt

In [10]:
#context / what it is
#tools available
#explains process roughly to give it room to think
#Let us know when it is finished, preferably with a keyword or mark
#LLMs are undeterministic so they may still not follow the prompt exactly. Some models are rated better at following directions. https://labs.scale.com/leaderboard/instruction_following.
# in general a bigger model or a reasoning model will be able to follow instructions better.

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

IMPORTANT: The word "DONE:" is a control signal, not a label. Never use it as a heading, section marker, or inline annotation. 
ONLY use the word "DONE:" as per the instructions below -- it has to come at the start of a reply.

You MUST gather information from at least 6 distinct sources before delivering your brief. 
If you have fewer than 6 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

When you are ready to deliver your final research brief, start your response with "DONE:" followed by the brief itself.
It is imperative that "DONE:" should be at the start of teh final response, so that it can be easily parsed and extracted.
You CANNOT and SHOULD NOT include "DONE:" in any other part of your response except at the very BEGINNING of the final research brief.

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

### Step 5: The Agentic Loop

In [11]:
def run_research_agent(topic: str, max_iterations:int=10) -> str:
    # """
    # Run the research agent on a topic and return the research brief.

    # Args:
    #     topic: the topic to research
    #     max_iterations: Safety limit to prevent infinite loops

    # Returns:
    #     The research brief as a string
    # """
    print(f"\n\U0001F50D starting research on: {topic}")
    print("=" * 60)

    messages = [{"role": "system", "content": RESEARCH_AGENT_PROMPT}, 
    {"role": "user", "content": f"Research the following topic and produce a compresive research brief: \n\n{topic}"}]
    iteration = 0

    while iteration < max_iterations:
        iteration += 1
        print(f"\n---Iteration {iteration} ---")

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        message = response.choices[0].message
        messages.append(message)

        if message.tool_calls:
            tool_results = handle_tool_calls(message.tool_calls)
            messages.extend(tool_results)  # This is a more concise way to add all tool results to messages
        else:
            result = message.content

            if result.startswith("DONE:"):
                research_brief = result[5:].strip()
                print(f"\n \u2705 Research complete!")
                return research_brief
            else:
                print(f"  \u001f Agent is thinking:")
                pprint(result)

        if (iteration == max_iterations - 1):
            print("  \u26a0 Safety limit reached. Stopping research in next iteration")
            messages.append({"role": "user", "content": "You have reached your maximum number of allowed iterations. You MUST return a brief on the next iteration"})

    messages.append("Unable to return a brief in 10 iterations")
    return "Research incomplete. Max iteration reached without returning a brief within iteration limits"

In [12]:
# brief = run_research_agent("AI in Healthcare in 2030")
# display(Markdown(brief))

### Evals

Prompt for Claude Vibe Coding

"Please write an LLM-as-a-judge prompt for testing the following failure: Research Agent failed to use at least  6 different sources in the deliverable research brief. 

Note: we're not testing correct formal citation, rather we're simply testing the breadth of research by checking if the Agent referenced 6 sources. 

Use the attached example as a reference on how I'd like the prompt structured"

In [13]:
JUDGE_PROMPT = """# Judge Prompt: Insufficient Source Breadth (TRUE/FALSE)

You are scoring a research agent **transcript/trace** to determine if there was an **insufficient source breadth failure**.
Return **only** `TRUE` or `FALSE`.

## Definitions

**Deliverable research brief:** The final written research output the agent presents to the user as its answer (not intermediate notes, scratchpad reasoning, or tool calls).

**Source:** A distinct, identifiable document, webpage, publication, report, dataset, or organization that the brief explicitly references (by URL, title, author, publisher, or clear name).

**Distinct source:** A source that is different from every other source already counted. Multiple references to the same document or page count once. Different URLs that point to the same underlying document count once.

**Formal citation is NOT required.** Do not evaluate citation formatting, style, or completeness. A source counts if a reader could reasonably tell which document/publication/organization the agent is referring to (e.g., an inline link, a name like "McKinsey's 2024 report," or a numbered reference list).

**Insufficient source breadth (label TRUE):** Any of the below occurred:

1. **Fewer than 6 distinct sources referenced in the brief:** The deliverable research brief references 5 or fewer distinct sources.
2. **No deliverable produced:** The agent never produced a research brief, so no sources are referenced.
3. **Sources only in the trace, not the brief:** The agent searched, opened, or retrieved additional sources during research, but they are not referenced in the final brief. These do not count toward the 6.
4. **Duplicate or inflated source counts:** The brief reaches 6 or more references only by repeating the same source, listing multiple URLs to the same document, or counting sections of one document as separate sources.
5. **Unattributed claims:** The brief makes claims attributed only to vague or unnamed sources (e.g., "studies show," "experts say," "according to research") with no identifiable source. These do not count toward the 6.

**No insufficient source breadth (label FALSE):** The deliverable research brief explicitly references **6 or more distinct, identifiable sources**.

## Scoring Instructions

1. Locate the final deliverable research brief in the transcript/trace.
2. List each distinct, identifiable source referenced in the brief, merging duplicates.
3. Exclude sources that appear only in tool calls, search results, or intermediate reasoning.
4. If the count is 6 or more, return `FALSE`. If the count is 5 or fewer, return `TRUE`.

## Output Format

Return exactly one token: `TRUE` or `FALSE`. No explanations."""

In [14]:
TOPICS = [
    "Impact of GLP-1 drugs on the global food industry",
    "Solid-state battery commercialization timeline",
    "Causes and consequences of the 2008 financial crisis",
    "Effectiveness of four-day work weeks",
    "History and current state of CRISPR gene therapies",
    "Microplastics in drinking water: health risks",
    "Rise of Southeast Asia as a manufacturing hub",
    "Ethical debates around AI-generated art",
    "Urban vertical farming economics",
    "Origins and spread of the Bronze Age collapse",
]

In [18]:
import re
import traceback
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
import pandas as pd
import contextlib
import io

client = OpenAI(timeout=60, max_retries=2)  # reads OPENAI_API_KEY from env

def judge(brief: str) -> dict:
    out = client.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": f"<transcript>\n{brief}\n</transcript>"},
        ],
    ).choices[0].message.content
    label = re.findall(r"LABEL:\s*\**\s*(TRUE|FALSE)", out, re.I)
    count = re.findall(r"COUNT:\s*(\d+)", out)
    return {
        "failure": label[-1].upper() == "TRUE" if label else None,  # last match wins
        "n_sources": int(count[-1]) if count else None,
        "judge_output": out,
    }

def evaluate(topic: str) -> dict:
    try:
        print(f"▶ started: {topic}", flush=True)
        with contextlib.redirect_stdout(io.StringIO()):
            brief = run_research_agent(topic)
        res = {"topic": topic, **judge(brief), "brief": brief}
        print(f"✔ done: {topic} | sources={res['n_sources']} failure={res['failure']}", flush=True)
        return res
    except Exception:
        err = traceback.format_exc()
        print(f"✖ error: {topic}\n{err}", flush=True)
        return {"topic": topic, "failure": None, "n_sources": None, "judge_output": "", "brief": f"ERROR: {err}"}

with ThreadPoolExecutor(max_workers=5) as ex:
    futures = [ex.submit(evaluate, t) for t in TOPICS]
    results = [f.result() for f in as_completed(futures)]

df = pd.DataFrame(results)
print(df[["topic", "n_sources", "failure"]].to_string(index=False))
valid = df["failure"].dropna()
print(f"\nFailure rate: {valid.mean():.0%} ({int(valid.sum())}/{len(valid)}); errors/unparsed: {df['failure'].isna().sum()}")

### Eval Best Practices
1) Code-based where possible 
2) Anaylize traces as well, not just output/results
3) Inovlve the product person, does the AI product work correctly (product manager)
4) One judge per failure mode. A seperate LLM per criteria you are testing ("Did it meet...")
5) Binary result or grading scale 0-5. Binary is best because grading becomes relative.
6) Run evals across diverse inputs